In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# The Halting Problem

This notebook proves the undecidability of the *halting problem*.
In this proof, programs are represented as *strings*.  Hence we need a way to turn a string into a function that we can call.  In <samp>Python</samp>, this is done by the built-in function `exec`.  <samp>OCaml</samp> is a compiled language and has no such function.  However, the toplevel that executes this notebook can execute <samp>OCaml</samp> code that is given as a string (see the notebook `Eval.ipynb`).  Therefore, we start by implementing a function `exec` that passes a string to the toplevel.

The function `exec` takes one argument:
- `code` is a string containing <samp>OCaml</samp> code, e.g. the definition of a function.

If `code` can be compiled without errors, then `code` is executed and all definitions contained in `code` become available in the following cells of this notebook.  Otherwise, the error message of the compiler is displayed and the message *"This is not a valid <samp>OCaml</samp> definition."* is printed.  The function does not return a meaningful result.

**Implementation:** The function `Toploop.use_input` executes the phrases contained in `code` as if they had been typed into the toplevel.  The answers of the toplevel, e.g. the types of the defined functions, and the error messages are written to the formatter `Format.std_formatter`, i.e. to the standard output.  The function `Toploop.use_input` returns `false` if an error has occurred.  The module `Toploop` belongs to the compiler libraries, which are added to the search path by the file `setup.ml`.

In [ ]:
let exec (code : string) : unit =
  if not (Toploop.use_input Format.std_formatter (Toploop.String code)) then
    print_endline "This is not a valid OCaml definition."

We continue with a number of definitions.

A string $t$ is a *test function with name $f$*
iff $t$ has the form
```
{|
let f (x : string) =
  body
|}
```
and, furthermore, the string $t$ can be compiled as an <samp>OCaml</samp> function, that is the evaluation of
```
exec t
```
does not yield an error.  Note that in <samp>OCaml</samp>, compiling a function includes *type checking*.  Hence a string that is syntactically correct but contains a type error is not a test function.  Furthermore, by definition the argument `x` of a test function is a string.
The set of all test functions is denoted as $T\!F$.  If $t \in T\!F$ and $t$ has the name $f$, then
this is written as $\mathtt{name}(t) = f$.

We provide some examples next.  We use *quoted strings*, i.e. strings that start with `{|` and end with `|}`.  These strings can extend over several lines and can contain double quotes.

The string `t1` is a *test function* with the name `simple`.

In [ ]:
let t1 = {|
let simple (x : string) =
  0
|}

In [ ]:
print_endline t1

By executing this string, we define the function `simple`.

In [ ]:
exec t1

In [ ]:
simple "1"

The string `t2` is a *test function* with the name `loop`.  The toplevel infers the type `string -> 'a` for `loop`: As the loop `while true do ... done` never terminates, it never produces a value.  Therefore, the type checker allows it to have any type `'a`.

In [ ]:
let t2 = {|
let loop (x : string) =
  let n = ref 0 in
  while true do
    n := !n + 1
  done
|}

In [ ]:
exec t2

Obviously, the next cell will not terminate.  You have to interrupt the kernel to stop it.

In [ ]:
loop "1"

The string `t3` is not a *test function* since it is a <samp>C</samp> function, but not an <samp>OCaml</samp> function.

In [ ]:
let t3 = {|
int hugo(int x) {
    return ++x;
}
|}

As `t3` is not a test function, the next cell will yield a *syntax error*.

In [ ]:
exec t3

The string `t4` is not a test function either.  It is syntactically correct, but it contains a *type error*: the argument `x` is a string, but the operator `-` can only be applied to integers.

In [ ]:
let t4 = {|
let otto (x : string) =
  x - 1
|}

In [ ]:
exec t4

**Notation:**
If $n$ is the name of an <samp>OCaml</samp> function that takes $k$ arguments $a_1$, $\cdots$, $a_k$,
then we write
$$n(a_1, \cdots, a_k) \leadsto r$$
iff the evaluation of the expression $n\;a_1\;\cdots\;a_k$ yields the result $r$.

If we are not
concerned with the result $r$ but only want to state that the evaluation *terminates* eventually,
then we will write
$$n(a_1, \cdots, a_k) \,\downarrow$$
and read this notation as *evaluation of $n(a_1, \cdots, a_k)$ terminates*.

If the evaluation of the expression $n(a_1, \cdots, a_k)$ does **not** terminate, this is
written as
$$n(a_1, \cdots, a_k) \,\uparrow.$$
This notation is read as *evaluation of $n(a_1, \cdots, a_k)$ diverges*.

The *halting problem* for <samp>OCaml</samp> functions is the question whether there is an
<samp>OCaml</samp> function
```
let stops (t : string) (a : string) : int =
  ...
```
that takes as input a string $t$ and a string $a$ and that satisfies the following specification:
- $t \not\in T\!F \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 2$.

  If the first argument of `stops` is not a test function, then
  `stops t a = 2`.
- $t \in T\!F \,\wedge\, \mathtt{name}(t) = n \,\wedge\, n(a)\downarrow \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 1$.

  If the first argument of `stops` is a test function with name $n$ and, furthermore,
  the evaluation of $n(a)$ terminates, then `stops t a = 1`.

- $t \in T\!F \,\wedge\, \mathtt{name}(t) = n \,\wedge\, n(a)\uparrow \quad\Leftrightarrow\quad \mathtt{stops}(t, a) \leadsto 0$.

  If the first argument of `stops` is a test function with name $n$ but the evaluation of $n(a)$
  diverges, then `stops t a = 0`.

The string `turing` is a test function with name `alan`.
It makes use of the function `stops`.  The function `alan` takes one argument:
- `x` is a string.

It computes `stops x x`.  If the result is `1`, then `alan` enters an infinite loop.  Otherwise, `alan` returns the result of `stops x x`.

In [ ]:
let turing = {|
let alan (x : string) : int =
  let result = stops x x in
  if result = 1 then
    while true do
      print_endline "... looping ..."
    done;
  result
|}

As `turing` is a test function, there are only two possibilities for the call `stops turing turing`.
1. Case:  `stops turing turing = 1`.
   Provided the function `stops` solves the *halting problem*, then the call `alan turing` should terminate.
2. Case:  `stops turing turing = 0`.
   According to the specification of `stops`, then the call `alan turing` should diverge.

Let us check both cases.  First, we pretend that `stops turing turing = 1`.

In [ ]:
let stops (t : string) (a : string) : int =
  1

Now we can execute the string `turing`.  In <samp>OCaml</samp>, a function can only use functions that have already been defined.  Hence, we had to define `stops` before executing `turing`.

In [ ]:
exec turing

We will see that contrary to the specification of `stops`, the call `alan turing` loops.

**Warning:** The next cell does not terminate.  It prints the line `... looping ...` forever.  You have to interrupt the kernel to stop it.  As the kernel first has to deliver all lines that have been printed so far, the output continues for some seconds after the interruption.

In [ ]:
alan turing

Let us assume that `stops turing turing = 0` next.

In [ ]:
let stops (t : string) (a : string) : int =
  0

In <samp>OCaml</samp>, names are bound *statically*:  The function `alan` defined above uses the version of `stops` that was visible when `alan` was defined.  This version of `stops` always returns `1`.  The new definition of `stops` creates a new function that *shadows* the old one, but `alan` still refers to the old function.  Therefore, we have to execute the string `turing` again, so that `alan` uses the new version of `stops`.

In [ ]:
exec turing

According to the specification of the function `stops`, the call `alan turing` should loop.  However, it returns `0` instead.

In [ ]:
alan turing

We can only conclude that there is no way that the function `stops` can satisfy its specification.
Hence the *halting problem* is unsolvable.